# EDSS 전체 재구성 스키마 사전검사

## tl;dr

전체 233개 논리 테이블, 265개 물리 단위, 278개 ZIP을 출력 패널 생성 없이 전수검사했다. CSV 멤버 1,142개와 데이터 행 180,119,183개가 모두 읽혔고, 행 폭 불일치와 읽기 실패는 0건이다. 제공 연도와 실제 관찰 연도가 다른 물리 단위 4개는 후속 패널 메타데이터에 그대로 보존해야 한다.

## Context & Methods

전체 패널을 만들기 전에 인벤토리의 명시적 ZIP 경로를 사용해 ZIP·중첩 ZIP·CSV를 스트리밍으로 읽고, 물리 `domnCd`별 행 수, 헤더, 인코딩, 관찰 연도와 행 폭을 검사한다.

### Key Assumptions

- EDSS 실시간 다운로드 목록의 `domnCd`가 물리 입력 단위다.
- `(분야, 카탈로그 코드, 테이블명)`이 논리 테이블을 정의한다.
- 제공 연도와 관찰 연도 차이는 자동 보간하거나 삭제하지 않는다.
- 취업통계 2023~2024년 구조에 없는 `개방ID`는 추정하지 않는다.

## Data

- `data/metadata/edss_full_rebuild_inventory.csv`
- `data/metadata/edss_full_rebuild_schema_scan.jsonl`
- `data/metadata/edss_full_rebuild_schema_scan_summary.json`

원본 ZIP과 대용량 패널은 노트북에 복제하지 않는다.

In [1]:
import json
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
metadata = repo_root / "data/metadata"
summary = json.loads((metadata / "edss_full_rebuild_schema_scan_summary.json").read_text(encoding="utf-8"))
profiles = [
    json.loads(line)
    for line in (metadata / "edss_full_rebuild_schema_scan.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
len(profiles), summary["verification"]

(265,
 {'issue_count': 4,
  'issue_counts_by_severity': {'medium': 4},
  'issue_counts_by_type': {'advertised_observed_year_difference': 4},
  'scan_complete': True,
  'panel_build_ready': True})

## Results

In [2]:
assert summary["logical_table_count"] == 233
assert summary["physical_unit_count"] == 265
assert summary["scanned_physical_unit_count"] == 265
assert summary["archive_count"] == 278
assert summary["csv_member_count"] == 1142
assert summary["row_count"] == 180_119_183
assert summary["malformed_row_count"] == 0
assert summary["verification"]["scan_complete"] is True
assert summary["verification"]["panel_build_ready"] is True
assert len(profiles) == 265
assert sum(profile["total_rows"] for profile in profiles) == summary["row_count"]

{
    "논리 테이블": summary["logical_table_count"],
    "물리 단위": summary["physical_unit_count"],
    "ZIP": summary["archive_count"],
    "CSV 멤버": summary["csv_member_count"],
    "행": summary["row_count"],
    "행 폭 오류": summary["malformed_row_count"],
    "스키마 변형 논리 테이블": summary["logical_tables_with_schema_variants"],
}

{'논리 테이블': 233,
 '물리 단위': 265,
 'ZIP': 278,
 'CSV 멤버': 1142,
 '행': 180119183,
 '행 폭 오류': 0,
 '스키마 변형 논리 테이블': 58}

In [3]:
[
    {
        "분야": issue["source"],
        "코드": issue["catalog_code"],
        "테이블": issue["dataset"],
        "제공 연도": issue["advertised_years"],
        "관찰 연도": ", ".join(issue["observed_years"]),
        "누락 제공 연도": ", ".join(issue["missing_advertised_years"]),
        "추가 관찰 연도": ", ".join(issue["unexpected_observed_years"]),
    }
    for issue in summary["issues"]
]

[{'분야': '고등교육통계',
  '코드': '0222',
  '테이블': '재적학생현황_대학원_공동운영',
  '제공 연도': '2010~2023',
  '관찰 연도': '2010, 2011, 2012, 2013',
  '누락 제공 연도': '2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023',
  '추가 관찰 연도': ''},
 {'분야': '고등교육통계',
  '코드': '0314',
  '테이블': '교원급여및수업시수현황',
  '제공 연도': '2009~2014',
  '관찰 연도': '2009, 2010, 2011, 2012, 2013',
  '누락 제공 연도': '2014',
  '추가 관찰 연도': ''},
 {'분야': '고등교육통계',
  '코드': '0317',
  '테이블': '교원현황_산업대학',
  '제공 연도': '2014~2022',
  '관찰 연도': '2014, 2015, 2016, 2020, 2021, 2022',
  '누락 제공 연도': '2017, 2018, 2019',
  '추가 관찰 연도': ''},
 {'분야': '고등교육통계',
  '코드': '0405',
  '테이블': '사이버강좌개설및수강현황_전문대학',
  '제공 연도': '2014~2022',
  '관찰 연도': '2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025',
  '누락 제공 연도': '',
  '추가 관찰 연도': '2023, 2024, 2025'}]

In [4]:
employment = [
    {
        "domnCd": profile["domn_code"],
        "제공 연도": profile["advertised_years"],
        "관찰 행": profile["total_rows"],
        "원본 열": profile["original_column_count"],
        "개방ID 있음": profile["has_open_id"],
        "학교명 있음": "학교명" in profile["original_fields"],
        "학과명 있음": "학과명" in profile["original_fields"],
    }
    for profile in profiles
    if profile["source"] == "취업통계"
]
assert employment == [
    {"domnCd": "13299", "제공 연도": "2010~2022", "관찰 행": 7_277_987, "원본 열": 146, "개방ID 있음": True, "학교명 있음": False, "학과명 있음": True},
    {"domnCd": "13300", "제공 연도": "2023~2024", "관찰 행": 46_962, "원본 열": 24, "개방ID 있음": False, "학교명 있음": True, "학과명 있음": True},
]
employment

[{'domnCd': '13299',
  '제공 연도': '2010~2022',
  '관찰 행': 7277987,
  '원본 열': 146,
  '개방ID 있음': True,
  '학교명 있음': False,
  '학과명 있음': True},
 {'domnCd': '13300',
  '제공 연도': '2023~2024',
  '관찰 행': 46962,
  '원본 열': 24,
  '개방ID 있음': False,
  '학교명 있음': True,
  '학과명 있음': True}]

## Takeaways

- 180,119,183행 전체에서 CSV 행 폭 오류나 읽기 실패가 없어 패널 빌드를 시작할 수 있다.
- 58개 논리 테이블에는 연도 또는 물리 단위 사이의 스키마 변형이 있으므로 열 합집합과 열별 적용 연도를 보존해야 한다.
- 제공·관찰 연도 차이 4건은 빈 연도를 생성하지 말고 원본 범위 차이로 기록한다.
- 취업통계 `학생인적취업정보`는 2010~2022년에 `개방ID`가 있지만 2023~2024년에는 학교명·학과명만 있고 코드 열이 없다. 두 구조를 ID로 강제 연결하지 않는다.